# 01h — Extract SANLC Land Cover Context
**Data source:** [South African National Land Cover (SANLC) 2020 & 2022](https://egis.environment.gov.za/)

**Input:** 
- `train_base.parquet`, `val_base.parquet` — from notebook 00 output
- `SA_NLC_2022_ALBERS.tif` + `.vat.dbf` — manual download, upload as Kaggle input
- `SA_NLC_2020_ALBERS.tif` + `.vat.dbf` — manual download, upload as Kaggle input

**Output:** `sanlc.parquet` (one row per unique station)

**Estimated time:** ~5 min (if raster files are uploaded)

> **Note:** SANLC files are manual downloads from the South African EGIS portal. If they are not present in your Kaggle inputs, this notebook will gracefully output placeholder/NaN columns to prevent downstream pipeline crashes.

In [ ]:
# Install required packages if running in Kaggle environment
!pip install -q geopandas pyarrow requests tqdm fiona pyogrio rasterio shapely

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from shapely.geometry import Point
import matplotlib.pyplot as plt
import os, time, logging
import warnings
warnings.filterwarnings('ignore')

# === Logging Setup ===
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-5s | %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('01h_sanlc')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'axes.titleweight': 'bold', 'font.size': 11})

OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Column config
LAT_COL     = 'Latitude'
LON_COL     = 'Longitude'
STATION_COL = 'station_id'

# Helper function to find input file path in Kaggle
def find_file(filename, default_dir='/kaggle/working'):
    target = os.path.join(default_dir, filename)
    if os.path.exists(target):
        return target
    input_dir = '/kaggle/input'
    if os.path.exists(input_dir):
        for root, _, files in os.walk(input_dir):
            if filename in files:
                return os.path.join(root, filename)
    raise FileNotFoundError(f"File {filename} not found in input/working directories.")

In [ ]:
# Load base data to get unique stations
try:
    base_train_path = find_file('train_base.parquet')
    base_val_path   = find_file('val_base.parquet')
    train_base = pd.read_parquet(base_train_path)
    val_base   = pd.read_parquet(base_val_path)
    all_data   = pd.concat([train_base, val_base], ignore_index=True)
    unique_stations = all_data.groupby(STATION_COL)[[LAT_COL, LON_COL]].first().reset_index()
    log.info(f'Loaded unique stations: {len(unique_stations)}')
except FileNotFoundError as e:
    log.error(e)
    unique_stations = pd.DataFrame()  # empty

---
## SANLC Extraction Helpers

In [ ]:
def load_raster_attribute_table(vat_dbf_path):
    """Loads Value Attribute Table (VAT) DBF file and returns mapping dictionary."""
    try:
        dbf_df = gpd.read_file(vat_dbf_path)
        val_col = 'Value' if 'Value' in dbf_df.columns else dbf_df.columns[0]
        class_col = 'LC_Class' if 'LC_Class' in dbf_df.columns else dbf_df.columns[1]
        return dict(zip(dbf_df[val_col], dbf_df[class_col]))
    except Exception as e:
        log.warning(f'Failed to load VAT dbf {vat_dbf_path}: {e}')
        return {}

def fetch_raster_mean_value(lat, lon, raster_src, buffer_m=1000):
    """Extract mean raster value within buffer around coordinates."""
    point = Point(lon, lat)
    deg_buffer = buffer_m / 111320.0  # approximate meters to degrees conversion
    buffered_geom = point.buffer(deg_buffer)
    
    try:
        # Crop raster to buffered area
        masked_img, _ = mask(raster_src, [buffered_geom], crop=True, nodata=0)
        valid_pixels = masked_img[0][masked_img[0] > 0]
        return float(valid_pixels.mean()) if valid_pixels.size > 0 else 0.0
    except Exception:
        return 0.0

def fetch_mapped_class(lat, lon, raster_src, vat_mapping, buffer_m=1000):
    """Fetches numeric raster mean and maps to descriptive class name."""
    val = fetch_raster_mean_value(lat, lon, raster_src, buffer_m)
    if val == 0.0:
        return 'Unclassified/No Data'
    rounded_val = int(round(val))
    return vat_mapping.get(rounded_val, f'Class_{rounded_val}')

---
## Perform Extraction

In [ ]:
sanlc_df = unique_stations.copy()
sanlc_2022_extracted = False
sanlc_2020_extracted = False

# Check for SANLC 2022 files
try:
    tif_2022 = find_file('SA_NLC_2022_ALBERS.tif')
    dbf_2022 = find_file('SA_NLC_2022_ALBERS.tif.vat.dbf')
    
    log.info(f'Extracting SANLC 2022 using: {tif_2022}')
    mapping_2022 = load_raster_attribute_table(dbf_2022)
    with rasterio.open(tif_2022) as src:
        sanlc_df['sanlc2022_class_1km'] = sanlc_df.apply(
            lambda r: fetch_mapped_class(r[LAT_COL], r[LON_COL], src, mapping_2022, 1000), axis=1
        )
    sanlc_2022_extracted = True
    log.info('SANLC 2022 extraction completed successfully.')
except FileNotFoundError:
    log.warning('SA_NLC_2022 files not found in inputs. Creating placeholder column (NaN).')
    sanlc_df['sanlc2022_class_1km'] = np.nan

# Check for SANLC 2020 files
try:
    tif_2020 = find_file('SA_NLC_2020_ALBERS.tif')
    dbf_2020 = find_file('SA_NLC_2020_ALBERS.tif.vat.dbf')
    
    log.info(f'Extracting SANLC 2020 using: {tif_2020}')
    mapping_2020 = load_raster_attribute_table(dbf_2020)
    with rasterio.open(tif_2020) as src:
        sanlc_df['sanlc2020_class_1km'] = sanlc_df.apply(
            lambda r: fetch_mapped_class(r[LAT_COL], r[LON_COL], src, mapping_2020, 1000), axis=1
        )
    sanlc_2020_extracted = True
    log.info('SANLC 2020 extraction completed successfully.')
except FileNotFoundError:
    log.warning('SA_NLC_2020 files not found in inputs. Creating placeholder column (NaN).')
    sanlc_df['sanlc2020_class_1km'] = np.nan

display(sanlc_df.head())

---
## Save Output

In [ ]:
out_path = f'{OUTPUT_DIR}/sanlc.parquet'
sanlc_df.to_parquet(out_path, index=False)
size_kb = os.path.getsize(out_path) / 1024
log.info(f'Saved: {out_path} ({size_kb:.1f} KB, {len(sanlc_df)} rows)')
print('\n=== DONE ===')
print('Output: sanlc.parquet')
print(f'SANLC 2022 Status: {"Extracted" if sanlc_2022_extracted else "Placeholder/NaN"}')
print(f'SANLC 2020 Status: {"Extracted" if sanlc_2020_extracted else "Placeholder/NaN"}')
print('Next: add this notebook output as dataset input for 01e')